# Data Validation Tests

This notebook provides comprehensive validation of the ITD evaluation data.

**Purpose:**
- Validate YAML data structure and integrity
- Test answer values against question definitions
- Verify completeness of evaluations
- Explore edge cases and data quality

**Usage:**
- Run before committing new answer files
- Use as reference for adding new platforms
- Visual inspection of data quality

In [ ]:
# Setup
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

from utils.loader import load_questions, load_answers
from utils.validators import (
    validate_platform_results,
    validate_metadata,
    validate_answer_values,
    validate_question_coverage,
    ValidationError
)
from utils.scoring import calculate_platform_score
import pandas as pd

print("✓ Imports successful")

## Test 1: Load Questions

Verify that questions.yml is well-formed and contains all required fields.

In [ ]:
# Load questions
questions = load_questions()

print(f"Loaded {len(questions)} questions")
print(f"\nQuestion codes: {sorted(questions.keys())}")

# Verify structure
for code, q in questions.items():
    assert 'category' in q, f"{code}: missing category"
    assert 'text' in q, f"{code}: missing text"
    assert 'weight' in q, f"{code}: missing weight"
    assert 'answers' in q, f"{code}: missing answers"
    assert q['category'] in ['UGC', 'ADS'], f"{code}: invalid category"
    assert isinstance(q['weight'], (int, float)), f"{code}: weight must be numeric"
    
print("\n✓ All questions have required fields")

## Test 2: Question Weights Distribution

Analyze the distribution of question weights to ensure importance is balanced.

In [ ]:
import matplotlib.pyplot as plt

# Analyze weights by category
ugc_weights = [q['weight'] for q in questions.values() if q['category'] == 'UGC']
ads_weights = [q['weight'] for q in questions.values() if q['category'] == 'ADS']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Weight distribution
ax1.hist([ugc_weights, ads_weights], label=['UGC', 'ADS'], bins=5)
ax1.set_xlabel('Question Weight')
ax1.set_ylabel('Count')
ax1.set_title('Distribution of Question Weights')
ax1.legend()

# Category totals
categories = ['UGC', 'ADS']
totals = [sum(ugc_weights), sum(ads_weights)]
ax2.bar(categories, totals, color=['#2C5F8D', '#45B7D1'])
ax2.set_ylabel('Total Weight')
ax2.set_title('Maximum Possible Score by Category')

plt.tight_layout()
plt.show()

print(f"UGC max score: {sum(ugc_weights):.2f}")
print(f"ADS max score: {sum(ads_weights):.2f}")
print(f"Total max score: {sum(ugc_weights) + sum(ads_weights):.2f}")

## Test 3: Validate Reddit Brazil Data

Comprehensive validation of the Reddit Brazil evaluation.

In [ ]:
# Load Reddit Brazil data
answers = load_answers('reddit', 'BR')

print("Reddit Brazil Metadata:")
for key, value in answers['metadata'].items():
    print(f"  {key}: {value}")

# Test 3.1: Metadata validation
try:
    validate_metadata(answers['metadata'])
    print("\n✓ Metadata is valid")
except ValidationError as e:
    print(f"\n✗ Metadata validation failed: {e}")

# Test 3.2: Answer values validation
try:
    validate_answer_values(questions, answers['ugc_answers'], 'UGC')
    print("✓ UGC answer values are valid")
except ValidationError as e:
    print(f"✗ UGC validation failed: {e}")

try:
    validate_answer_values(questions, answers['ads_answers'], 'ADS')
    print("✓ ADS answer values are valid")
except ValidationError as e:
    print(f"✗ ADS validation failed: {e}")

# Test 3.3: Question coverage
try:
    validate_question_coverage(questions, answers)
    print("✓ All questions have answers")
except ValidationError as e:
    print(f"✗ Coverage validation failed: {e}")

## Test 4: Scoring Calculation Verification

Manually verify scoring calculations to ensure correctness.

In [ ]:
# Calculate scores
results = calculate_platform_score('reddit', 'BR')

# Manual verification of UGC scores
print("UGC Score Breakdown:")
print("=" * 60)
ugc_manual_total = 0

for detail in results['ugc_details']:
    code = detail['question_code']
    q_weight = detail['question_weight']
    a_weight = detail['answer_weight']
    score = detail['question_score']
    manual_score = q_weight * a_weight
    
    match = "✓" if abs(score - manual_score) < 0.01 else "✗"
    print(f"{match} {code}: {q_weight:.1f} × {a_weight:.1f} = {score:.2f}")
    
    ugc_manual_total += manual_score

print(f"\nCalculated UGC Total: {results['ugc_score']:.2f}")
print(f"Manual UGC Total:     {ugc_manual_total:.2f}")
print(f"Match: {abs(results['ugc_score'] - ugc_manual_total) < 0.01}")

print("\n" + "=" * 60)
print("ADS Score Breakdown:")
print("=" * 60)
ads_manual_total = 0

for detail in results['ads_details']:
    code = detail['question_code']
    q_weight = detail['question_weight']
    a_weight = detail['answer_weight']
    score = detail['question_score']
    manual_score = q_weight * a_weight
    
    match = "✓" if abs(score - manual_score) < 0.01 else "✗"
    print(f"{match} {code}: {q_weight:.1f} × {a_weight:.1f} = {score:.2f}")
    
    ads_manual_total += manual_score

print(f"\nCalculated ADS Total: {results['ads_score']:.2f}")
print(f"Manual ADS Total:     {ads_manual_total:.2f}")
print(f"Match: {abs(results['ads_score'] - ads_manual_total) < 0.01}")

## Test 5: Results Validation

Validate the final computed results using the validator.

In [ ]:
try:
    validate_platform_results(results)
    print("✓ Results validation passed")
    print(f"\nFinal Scores:")
    print(f"  UGC: {results['ugc_score']:.2f} / {results['ugc_max']:.2f} ({results['ugc_percentage']:.1f}%)")
    print(f"  ADS: {results['ads_score']:.2f} / {results['ads_max']:.2f} ({results['ads_percentage']:.1f}%)")
    print(f"  Total: {results['total_score']:.2f} / {results['total_max']:.2f} ({results['total_percentage']:.1f}%)")
except ValidationError as e:
    print(f"✗ Results validation failed: {e}")

## Test 6: Visual Score Summary

Create a visual summary of the evaluation results.

In [ ]:
# Create comparison table
summary_data = {
    'Category': ['UGC', 'ADS', 'Total'],
    'Actual Score': [
        results['ugc_score'],
        results['ads_score'],
        results['total_score']
    ],
    'Max Score': [
        results['ugc_max'],
        results['ads_max'],
        results['total_max']
    ],
    'Percentage': [
        results['ugc_percentage'],
        results['ads_percentage'],
        results['total_percentage']
    ]
}

df = pd.DataFrame(summary_data)
print(df.to_string(index=False))

# Create bar chart
fig, ax = plt.subplots(figsize=(10, 5))
categories = ['UGC', 'ADS']
percentages = [results['ugc_percentage'], results['ads_percentage']]
colors = ['#2C5F8D', '#45B7D1']

bars = ax.bar(categories, percentages, color=colors, alpha=0.7)
ax.axhline(y=75, color='green', linestyle='--', label='Excellent (75%)')
ax.axhline(y=50, color='orange', linestyle='--', label='Moderate (50%)')
ax.axhline(y=25, color='red', linestyle='--', label='Limited (25%)')

ax.set_ylabel('Score (%)')
ax.set_title('Reddit Brazil - Category Scores')
ax.set_ylim(0, 100)
ax.legend()

# Add value labels on bars
for bar, pct in zip(bars, percentages):
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height + 2,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## Summary

This notebook validates:
- ✓ Question structure and weights
- ✓ Answer data integrity
- ✓ Scoring calculations
- ✓ Results consistency

**Next Steps:**
1. Run this notebook before committing changes
2. Use as template for testing new platforms
3. Extend with additional edge case tests as needed